In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, T5ForConditionalGeneration
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')
import random
from tqdm import tqdm
from scipy import stats
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


def set_seed(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def load_and_prepare_reasonable_dataset(file_path, test_size=0.2, min_samples_per_author=50, max_authors=100):
    """
    Load Google Jam dataset with reasonable parameters
    """
    print("Loading dataset...")
    data = pd.read_csv(file_path)
    
    data = data.dropna(subset=['flines', 'username'])
    data['flines'] = data['flines'].astype(str)
    data = data[data['flines'].str.strip() != '']
    
    author_counts = data['username'].value_counts()
    
    valid_authors = author_counts[author_counts >= min_samples_per_author].index
    filtered_data = data[data['username'].isin(valid_authors)]
    
    if len(valid_authors) > max_authors:
        top_authors = author_counts.head(max_authors).index
        filtered_data = filtered_data[filtered_data['username'].isin(top_authors)]
    
    label_encoder = LabelEncoder()
    filtered_data['EncodedLabels'] = label_encoder.fit_transform(filtered_data['username'])
    num_classes = len(label_encoder.classes_)
    
    train_data, test_data = custom_stratified_split(filtered_data, test_size=test_size)
    
    return train_data, test_data, label_encoder, num_classes

def custom_stratified_split(data, test_size=0.2):
    """Custom stratified split that ensures proper distribution"""
    train_data = []
    test_data = []
    
    grouped = data.groupby('username')
    
    for author, group in grouped:
        group = group.sample(frac=1, random_state=42).reset_index(drop=True)
        
        n_test = max(1, int(len(group) * test_size))
        
        if len(group) - n_test < 2:
            n_test = max(1, len(group) - 2)
        
        # Split
        test_samples = group.iloc[:n_test]
        train_samples = group.iloc[n_test:]
        
        test_data.append(test_samples)
        train_data.append(train_samples)
    
    train_data = pd.concat(train_data, ignore_index=True)
    test_data = pd.concat(test_data, ignore_index=True)
    
    train_data = train_data.sample(frac=1, random_state=42).reset_index(drop=True)
    test_data = test_data.sample(frac=1, random_state=42).reset_index(drop=True)
    
    return train_data, test_data


class CodeT5Dataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        """
        Dataset for CodeT5+ with author classification task
        We'll use sequence classification approach
        """
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label_encoder = LabelEncoder()
        
        if 'EncodedLabels' not in dataframe.columns:
            self.df['EncodedLabels'] = self.label_encoder.fit_transform(dataframe['username'])
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Tokenize the code
        encoding = self.tokenizer(
            row["flines"],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(row["EncodedLabels"], dtype=torch.long)
        }


class CodeT5ForSequenceClassification(nn.Module):
    """
    CodeT5+ model adapted for sequence classification
    Uses the encoder outputs with a classification head
    """
    def __init__(self, model_name="Salesforce/codet5p-220m", num_classes=50):
        super().__init__()
        
        self.codet5 = T5ForConditionalGeneration.from_pretrained(model_name)
        
        for param in self.codet5.decoder.parameters():
            param.requires_grad = False
        
        hidden_size = self.codet5.config.d_model
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size, num_classes)
        )
        
        self.pooler = nn.Linear(hidden_size, hidden_size)
        self.pooler_activation = nn.Tanh()
        
    def forward(self, input_ids, attention_mask):
        encoder_outputs = self.codet5.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        last_hidden_state = encoder_outputs.last_hidden_state  
        
        attention_mask_expanded = attention_mask.unsqueeze(-1).float()
        pooled_output = (last_hidden_state * attention_mask_expanded).sum(dim=1) / attention_mask_expanded.sum(dim=1)
        
        pooled_output = self.pooler(pooled_output)
        pooled_output = self.pooler_activation(pooled_output)
        
        logits = self.classifier(pooled_output)
        
        return logits
    
    def save_pretrained(self, path):
        """Save model weights"""
        torch.save(self.state_dict(), path)
    
    def load_pretrained(self, path):
        """Load model weights"""
        self.load_state_dict(torch.load(path))

class EnhancedCodeT5Classifier(nn.Module):
    """
    Enhanced CodeT5+ classifier with attention mechanism
    """
    def __init__(self, model_name="Salesforce/codet5p-220m", num_classes=50):
        super().__init__()
        
        self.codet5 = T5ForConditionalGeneration.from_pretrained(model_name)
        
        for param in self.codet5.decoder.parameters():
            param.requires_grad = False
        
        hidden_size = self.codet5.config.d_model
        
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 2),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size * 2, hidden_size),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size, num_classes)
        )
        
    def forward(self, input_ids, attention_mask):
        encoder_outputs = self.codet5.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        hidden_states = encoder_outputs.last_hidden_state 
        
        attention_weights = self.attention(hidden_states)  
        attention_weights = torch.softmax(attention_weights, dim=1)
        
        context_vector = torch.sum(hidden_states * attention_weights, dim=1)  
        
        logits = self.classifier(context_vector)
        
        return logits


def train_epoch(model, dataloader, optimizer, criterion, device, epoch_num):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch_num}")
    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100*correct/total:.2f}%'
        })
    
    epoch_loss = total_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc

def evaluate_model(model, dataloader, device):
    """Evaluate model on test set"""
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy()
            
            logits = model(input_ids, attention_mask)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(labels)
            all_probs.append(probs.cpu().numpy())
    
    accuracy = accuracy_score(all_labels, all_preds)
    all_probs = np.vstack(all_probs) if all_probs else np.array([])
    
    return accuracy, all_preds, all_labels, all_probs


def run_codet5_experiment(num_seeds=10, model_size="220m"):
    """
    Run CodeT5+ experiment across multiple seeds
    
    Args:
        num_seeds: Number of random seeds to evaluate
        model_size: CodeT5+ model size ('220m', '770m', '2b', '6b', '16b')
    """
    
    print("\n" + "="*80)
    print(f"CodeT5+ AUTHORSHIP ATTRIBUTION EXPERIMENT")
    print(f"Model: CodeT5+ {model_size}")
    print("="*80)
    
    model_names = {
        "220m": "Salesforce/codet5p-220m",
        "770m": "Salesforce/codet5p-770m",
        "2b": "Salesforce/codet5p-2b",
        "6b": "Salesforce/codet5p-6b",
        "16b": "Salesforce/codet5p-16b"
    }
    
    if model_size not in model_names:
        print(f"Warning: Model size {model_size} not recognized. Using 220m.")
        model_size = "220m"
    
    model_name = model_names[model_size]
    print(f"Using model: {model_name}")
    
    SEEDS = [42, 123, 456, 789, 999, 111, 222, 333, 444, 555][:num_seeds]
    
    file_path = "/home/aman_swaraj/Downloads/Codelite/Other SE Tasks/Authorship_Attribution/gcj2020.csv"
    print("\nLoading dataset with Option 1 (Top 50 authors)...")
    
    train_data, test_data, label_encoder, num_classes = load_and_prepare_reasonable_dataset(
        file_path, 
        test_size=0.2,
        min_samples_per_author=2,  
        max_authors=1000
    )
    
    print(f"\nDataset Statistics:")
    print(f"  Number of authors: {num_classes}")
    print(f"  Training samples: {len(train_data)}")
    print(f"  Test samples: {len(test_data)}")
    print(f"  Number of seeds: {num_seeds}")
    
    all_results = []
    
    for seed_idx, seed in enumerate(SEEDS):
        print(f"\n{'='*60}")
        print(f"SEED {seed_idx+1}/{num_seeds}: {seed}")
        print(f"{'='*60}")
        
        set_seed(seed)
        
        print("\n1. Setting up CodeT5+ model and tokenizer...")
        
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        train_dataset = CodeT5Dataset(train_data, tokenizer)
        test_dataset = CodeT5Dataset(test_data, tokenizer)
        
        if model_size in ["2b", "6b", "16b"]:
            batch_size = 4  
        else:
            batch_size = 8
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
        
        print(f"   Initializing CodeT5+ {model_size} model...")
        model = EnhancedCodeT5Classifier(model_name=model_name, num_classes=num_classes)
        model = model.to(device)
        
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"   Total parameters: {total_params:,}")
        print(f"   Trainable parameters: {trainable_params:,}")
        
        print("\n2. Training CodeT5+ model...")
        
        if model_size in ["2b", "6b", "16b"]:
            epochs = 2 
            learning_rate = 1e-5
        else:
            epochs = 8
            learning_rate = 2e-5
        
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
        criterion = nn.CrossEntropyLoss()
        
        best_accuracy = 0
        train_accuracies = []
        train_losses = []
        
        for epoch in range(1, epochs + 1):
            epoch_loss, epoch_acc = train_epoch(
                model, train_loader, optimizer, criterion, device, epoch
            )
            
            train_losses.append(epoch_loss)
            train_accuracies.append(epoch_acc)
            
            print(f"   Epoch {epoch}: Loss = {epoch_loss:.4f}, Accuracy = {epoch_acc:.2f}%")
            
            test_accuracy, _, _, _ = evaluate_model(model, test_loader, device)
            print(f"   Test Accuracy: {test_accuracy:.4f}")
            
            if test_accuracy > best_accuracy:
                best_accuracy = test_accuracy
                best_model_path = f"/home/aman_swaraj/Downloads/Codelite/codet5_{model_size}_seed{seed}_best.pt"
                torch.save(model.state_dict(), best_model_path)
                print(f"   ✓ Saved best model (accuracy: {best_accuracy:.4f})")
        
        model.load_state_dict(torch.load(best_model_path))
        
        print("\n3. Final evaluation...")
        
        test_accuracy, predictions, true_labels, probabilities = evaluate_model(
            model, test_loader, device
        )
        
        class_report = classification_report(
            true_labels, predictions,
            target_names=label_encoder.classes_,
            output_dict=True,
            zero_division=0
        )
        
        macro_f1 = class_report['macro avg']['f1-score']
        weighted_f1 = class_report['weighted avg']['f1-score']
        
        seed_results = {
            'seed': seed,
            'model_size': model_size,
            'test_accuracy': test_accuracy,
            'macro_f1': macro_f1,
            'weighted_f1': weighted_f1,
            'train_accuracy_final': train_accuracies[-1] if train_accuracies else 0,
            'train_loss_final': train_losses[-1] if train_losses else 0,
            'epochs': epochs,
            'learning_rate': learning_rate,
            'best_model_path': best_model_path
        }
        
        all_results.append(seed_results)
        
        print(f"   Test Accuracy: {test_accuracy:.4f}")
        print(f"   Macro F1: {macro_f1:.4f}")
        print(f"   Weighted F1: {weighted_f1:.4f}")
        
        print(f"\n   Top 5 authors by F1-score:")
        author_f1_scores = []
        for i, author in enumerate(label_encoder.classes_):
            if str(i) in class_report:
                f1 = class_report[str(i)]['f1-score']
                author_f1_scores.append((author, f1))
        
        author_f1_scores.sort(key=lambda x: x[1], reverse=True)
        for author, f1 in author_f1_scores[:5]:
            print(f"     {author}: F1 = {f1:.4f}")
    
    print("\n" + "="*80)
    print("COMPREHENSIVE RESULTS ANALYSIS")
    print("="*80)
    
    results_df = pd.DataFrame(all_results)
    
    print("\nPerformance Across Seeds:")
    print("-" * 60)
    
    metrics_to_show = ['test_accuracy', 'macro_f1', 'weighted_f1', 'train_accuracy_final']
    
    for metric in metrics_to_show:
        mean_val = results_df[metric].mean()
        std_val = results_df[metric].std()
        min_val = results_df[metric].min()
        max_val = results_df[metric].max()
        
        print(f"{metric.replace('_', ' ').title():25s}: {mean_val:.4f} ± {std_val:.4f}")
        print(f"  Range: [{min_val:.4f}, {max_val:.4f}]")
        print()
    
    print("\n" + "-" * 60)
    print("STATISTICAL ANALYSIS")
    print("-" * 60)
    
    if len(results_df) > 1:
        accuracies = results_df['test_accuracy']
        
        cv = (accuracies.std() / accuracies.mean()) * 100
        
        t_stat, p_value = stats.ttest_1samp(accuracies, 0.5)
        
        print(f"Accuracy Consistency:")
        print(f"  Mean: {accuracies.mean():.4f}")
        print(f"  Std: {accuracies.std():.4f}")
        print(f"  Coefficient of Variation: {cv:.2f}%")
        print(f"  Range: [{accuracies.min():.4f}, {accuracies.max():.4f}]")
        print()
        print(f"Statistical Significance vs Random (0.5):")
        print(f"  t-statistic: {t_stat:.4f}")
        print(f"  p-value: {p_value:.6f}")
        
        if p_value < 0.05:
            print("  ✓ Significantly better than random (p < 0.05)")
        else:
            print("  ✗ Not significantly better than random")
    
    print("\n" + "-" * 60)
    print("GENERATING VISUALIZATIONS")
    print("-" * 60)
    
    try:
        plt.figure(figsize=(12, 8))
        
        plt.subplot(2, 2, 1)
        seeds = [f"Seed {s}" for s in results_df['seed']]
        x_pos = np.arange(len(seeds))
        
        bars = plt.bar(x_pos, results_df['test_accuracy'], color='#4A90E2', alpha=0.8)
        plt.axhline(y=results_df['test_accuracy'].mean(), color='red', 
                   linestyle='--', linewidth=2, label=f'Mean: {results_df["test_accuracy"].mean():.4f}')
        plt.xlabel('Random Seed')
        plt.ylabel('Test Accuracy')
        plt.title(f'CodeT5+ {model_size}: Accuracy Across {num_seeds} Seeds')
        plt.xticks(x_pos, seeds)
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        for bar, acc in zip(bars, results_df['test_accuracy']):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{acc:.4f}', ha='center', va='bottom', fontsize=9)
        
        plt.subplot(2, 2, 2)
        x = np.arange(len(results_df))
        width = 0.25
        
        bars1 = plt.bar(x - width, results_df['test_accuracy'], width, 
                       label='Accuracy', color='#4A90E2', alpha=0.8)
        bars2 = plt.bar(x, results_df['macro_f1'], width, 
                       label='Macro F1', color='#50E3C2', alpha=0.8)
        bars3 = plt.bar(x + width, results_df['weighted_f1'], width, 
                       label='Weighted F1', color='#F5A623', alpha=0.8)
        
        plt.xlabel('Seed')
        plt.ylabel('Score')
        plt.title('Performance Metrics Comparison')
        plt.xticks(x, seeds)
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.subplot(2, 2, 3)
        
        plt.subplot(2, 2, 4)
        plt.boxplot(results_df['test_accuracy'])
        plt.ylabel('Accuracy')
        plt.title('Accuracy Distribution Across Seeds')
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'codet5_{model_size}_results.png', dpi=300, bbox_inches='tight')
        print(f"  Saved: codet5_{model_size}_results.png")
        
        plt.figure(figsize=(10, 6))
        
        metrics = ['test_accuracy', 'macro_f1', 'weighted_f1']
        colors = ['#4A90E2', '#50E3C2', '#F5A623']
        
        for idx, (metric, color) in enumerate(zip(metrics, colors)):
            plt.plot(range(1, len(results_df) + 1), results_df[metric], 
                    marker='o', linewidth=2, markersize=8, color=color, 
                    label=metric.replace('_', ' ').title())
        
        plt.xlabel('Seed Index')
        plt.ylabel('Score')
        plt.title(f'CodeT5+ {model_size}: Performance Metrics Trend')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.xticks(range(1, len(results_df) + 1))
        
        plt.tight_layout()
        plt.savefig(f'codet5_{model_size}_metrics_trend.png', dpi=300, bbox_inches='tight')
        print(f"  Saved: codet5_{model_size}_metrics_trend.png")
        
        plt.show()
        
    except Exception as e:
        print(f"  Visualization error: {e}")
    
    print("\n" + "-" * 60)
    print("SAVING RESULTS")
    print("-" * 60)
    
    results_path = f"/home/aman_swaraj/Downloads/Codelite/codet5_{model_size}_results.csv"
    results_df.to_csv(results_path, index=False)
    print(f"Detailed results saved to: {results_path}")
    
    summary_stats = {
        'metric': ['Test Accuracy', 'Macro F1', 'Weighted F1', 'Train Accuracy (final)'],
        'mean': [
            results_df['test_accuracy'].mean(),
            results_df['macro_f1'].mean(),
            results_df['weighted_f1'].mean(),
            results_df['train_accuracy_final'].mean()
        ],
        'std': [
            results_df['test_accuracy'].std(),
            results_df['macro_f1'].std(),
            results_df['weighted_f1'].std(),
            results_df['train_accuracy_final'].std()
        ],
        'min': [
            results_df['test_accuracy'].min(),
            results_df['macro_f1'].min(),
            results_df['weighted_f1'].min(),
            results_df['train_accuracy_final'].min()
        ],
        'max': [
            results_df['test_accuracy'].max(),
            results_df['macro_f1'].max(),
            results_df['weighted_f1'].max(),
            results_df['train_accuracy_final'].max()
        ]
    }
    
    summary_df = pd.DataFrame(summary_stats)
    summary_path = f"/home/aman_swaraj/Downloads/Codelite/codet5_{model_size}_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"Summary statistics saved to: {summary_path}")


if __name__ == "__main__":
    print("CodeT5+ Authorship Attribution Experiment")
    print("="*80)
    
    print("\nSelect experiment type:")
    print("1. Single CodeT5+ model evaluation (multiple seeds)")
    print("2. CodeT5+ model size comparison")
    print("="*40)
    
    choice = input("Enter your choice (1 or 2): ")
    
    if choice == "2":
        run_codet5_comparison_experiment()
    else:
        print("\nSelect CodeT5+ model size:")
        print("220m  - Small (fastest, good for testing)")
        print("770m  - Medium (balanced)")
        print("2b    - Large (better accuracy, slower)")
        print("6b    - Very large (needs significant memory)")
        print("16b   - Extra large (research use only)")
        print("="*40)
        
        model_size = input("Enter model size (default: 220m): ").strip() or "220m"
        num_seeds = int(input("Enter number of seeds (default: 3): ").strip() or "10")
        
        results_df, summary_df = run_codet5_experiment(
            num_seeds=num_seeds, 
            model_size=model_size
        )
        
        print("\n" + "="*80)
        print("EXPERIMENT COMPLETE - FINAL SUMMARY")
        print("="*80)
        print(f"Model: CodeT5+ {model_size}")
        print(f"Number of authors: {500}")  
        print(f"Training samples: {len(pd.read_csv('/home/aman_swaraj/Downloads/Codelite/Other SE Tasks/Authorship_Attribution/gcj2020.csv'))} (approx)")
        print(f"Number of seeds evaluated: {num_seeds}")
        print(f"Average test accuracy: {results_df['test_accuracy'].mean():.4f}")
        print(f"Accuracy range: [{results_df['test_accuracy'].min():.4f}, {results_df['test_accuracy'].max():.4f}]")
        print(f"Best seed: {int(results_df.loc[results_df['test_accuracy'].idxmax(), 'seed'])}")
        print(f"Worst seed: {int(results_df.loc[results_df['test_accuracy'].idxmin(), 'seed'])}")